# Feature Engineering
## Last-Mile Delivery Operations

Creates analytically-justified derived variables on top of the cleaned
dataset. Every feature here has a stated business/analytical purpose --
see `src/analysis/features.py` for the full rationale behind each one.

Not created: a peak/off-peak indicator. The dataset has no time-of-day
or timestamp field, so there is no basis to classify deliveries as
"peak" vs "off-peak" without inventing data that doesn't exist.

In [1]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from config.config import PROCESSED_DATA_FILE, PROCESSED_DATA_DIR
from src.analysis.features import engineer_features

df = pd.read_csv(PROCESSED_DATA_FILE)
df.shape

(25000, 16)

## Run the feature engineering pipeline

In [2]:
df_feat = engineer_features(df)
print("Shape before:", df.shape)
print("Shape after:", df_feat.shape)
df_feat.columns.tolist()

Shape before: (25000, 16)
Shape after: (25000, 24)


['row_id',
 'delivery_id',
 'delivery_partner',
 'package_type',
 'vehicle_type',
 'delivery_mode',
 'region',
 'weather_condition',
 'distance_km',
 'package_weight_kg',
 'delayed',
 'delivery_status',
 'delivery_rating',
 'delivery_cost',
 'delivery_time_hours_clean',
 'expected_time_hours_clean',
 'delay_duration_hours',
 'delay_category',
 'on_time_flag',
 'delivery_success_flag',
 'distance_category',
 'cost_per_km',
 'satisfaction_category',
 'performance_score']

## 1. `delay_duration_hours` and `delay_category`

Derived from the two cleaned time columns.

In [3]:
print(df_feat['delay_duration_hours'].describe())
print()
print(df_feat['delay_category'].value_counts())

count    25000.000000
mean        -6.859640
std          8.069482
min        -24.000000
25%        -14.000000
50%         -6.000000
75%          0.000000
max         12.000000
Name: delay_duration_hours, dtype: float64

delay_category
on_time_or_early    19534
mild_delay           2669
moderate_delay       2542
severe_delay          255
Name: count, dtype: int64


**Check:** 19,534 deliveries (78.1%) are `on_time_or_early`, with the
remainder split across mild (2,669), moderate (2,542), and severe (255)
delay tiers. This lines up with the 73.3% on-time rate from EDA -- the
gap (78.1% vs 73.3%) is expected, since `on_time_or_early` here includes
deliveries with `delay_duration_hours <= 0` even when `delivery_status`
is technically `failed`, since failed deliveries also carry a duration
value. This distinction is worth keeping in mind when using this
feature versus `on_time_flag` below, which is the stricter, status-based
version.

## 2. `on_time_flag` and `delivery_success_flag`

Binary flags for easy aggregation (mean = rate).

In [4]:
print("On-time rate:", df_feat['on_time_flag'].mean().round(4))
print("Success rate (not failed):", df_feat['delivery_success_flag'].mean().round(4))

On-time rate: 0.7332
Success rate (not failed): 0.9469


**Check:** `on_time_flag` mean = 0.7332, matching the 73.3% on-time
rate found in EDA. `delivery_success_flag` mean = 0.9469, meaning 94.7%
of deliveries eventually arrive (only 5.3% are outright failures) --
consistent with the failed rate found in EDA.

## 3. `distance_category`

Quartile-based bins from the dataset's own distribution.

In [5]:
df_feat['distance_category'].value_counts()

distance_category
short        6257
long         6257
medium       6244
very_long    6242
Name: count, dtype: int64

**Check:** near-even split across short/medium/long/very_long
(~6,240-6,260 each), as expected from a quartile-based cut. This makes
the category directly usable for fair group comparisons later, since no
tier is under- or over-represented.

## 4. `cost_per_km`

Distance-normalized cost, useful for comparing cost efficiency across segments.

In [6]:
df_feat['cost_per_km'].describe()

count    25000.000000
mean         7.041301
std          5.004036
min          5.002579
25%          5.420741
50%          5.744712
75%          6.494886
max         73.491667
Name: cost_per_km, dtype: float64

**Check:** median cost per km is ~₹5.74, but the max (~₹73.49) is far
above the 75th percentile (~₹6.49). This isn't a data error -- it
reflects short-distance deliveries where a largely fixed cost
(handling, base fee) gets divided by a very small distance, inflating
the per-km figure. This will be noted when using cost_per_km in
Business Analysis so short-distance outliers aren't misread as
"expensive routes."

## 5. `satisfaction_category`

Interpretable tiers from the 1-5 rating scale.

In [7]:
df_feat['satisfaction_category'].value_counts()

satisfaction_category
excellent    7387
good         7318
average      5773
poor         4522
Name: count, dtype: int64

**Check:** 58.8% of deliveries fall into good/excellent (ratings 4-5),
18.1% into poor (ratings 1-2). This distribution will let us compare
segments (e.g. "which weather condition has the highest 'poor' rate")
without doing per-rating-value comparisons.

## 6. `performance_score`

Composite 0-100 score: `on_time_flag * 40 + delivery_success_flag * 30
+ (rating / 5) * 30`. Weighted so failure is penalized most heavily,
lateness second, with rating as a confirming (not primary) signal --
avoids double-counting since rating already correlates strongly with
status.

In [8]:
print(df_feat['performance_score'].describe())
print()
print("Average performance_score by delivery_status:")
print(df_feat.groupby('delivery_status')['performance_score'].mean().round(2))

count    25000.000000
mean        79.732000
std         27.096351
min          6.000000
25%         48.000000
50%         94.000000
75%        100.000000
max        100.000000
Name: performance_score, dtype: float64

Average performance_score by delivery_status:
delivery_status
delayed      44.41
delivered    95.23
failed        7.84
Name: performance_score, dtype: float64


**Check:** the score cleanly separates the three outcomes --
`delivered` averages 95.2/100, `delayed` averages 44.4/100, `failed`
averages just 7.8/100. This confirms the score behaves as intended and
can be used as a single ranking metric in Business Analysis (e.g. to
rank partners or vehicle types on one number instead of three separate
rates).

## Save the feature-engineered dataset

In [9]:
output_path = PROCESSED_DATA_DIR / "delivery_logistics_features.csv"
df_feat.to_csv(output_path, index=False)
print(f"Saved feature-engineered dataset to: {output_path}")
print(f"Final shape: {df_feat.shape}")

Saved feature-engineered dataset to: E:\Python projects\last-mile-delivery-analysis\data\processed\delivery_logistics_features.csv
Final shape: (25000, 24)


## Feature Engineering Summary

| Feature | Type | Purpose |
|---|---|---|
| `delay_duration_hours` | numeric | Raw magnitude of lateness/earliness |
| `delay_category` | categorical | Interpretable delay severity tiers |
| `on_time_flag` | binary | Easy aggregation of on-time rate |
| `delivery_success_flag` | binary | Easy aggregation of completion rate |
| `distance_category` | categorical | Fair, evenly-populated distance comparison |
| `cost_per_km` | numeric | Distance-normalized cost efficiency |
| `satisfaction_category` | categorical | Interpretable rating tiers |
| `performance_score` | numeric (0-100) | Single composite ranking metric |

**Intentionally not created:** peak/off-peak indicator (no time-of-day
data available -- documented in Step 4's scope limitations).

Output saved to `data/processed/delivery_logistics_features.csv`,
ready for Business Analysis (Step 9).